# Google Play Store — Data Cleaning Pipeline (FINAL)
### Section A | Group 13

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/Google-Playstore-raw.csv'
CLEAN_PATH = '../data/Google-Playstore-Cleaned.csv'

df = pd.read_csv(RAW_PATH)

print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'Total nulls: {df.isnull().sum().sum():,}')

Loaded: 2,312,944 rows, 24 columns
Total nulls: 1,305,751


---
## Step 1: Drop Missing App Names

In [2]:
df.dropna(subset=['App Name'], inplace=True)
print(f'Remaining rows: {len(df):,}')

Remaining rows: 2,312,939


---
## Step 2: Rating & Rating Count

In [3]:
rating_mean = df['Rating'].mean()
df['Rating'] = df['Rating'].fillna(rating_mean)

rating_count_median = df['Rating Count'].median()
df['Rating Count'] = df['Rating Count'].fillna(rating_count_median)

print(f'Rating mean used: {rating_mean:.4f}')
print(f'Rating Count median used: {rating_count_median}')

Rating mean used: 2.2031
Rating Count median used: 6.0


---
## Step 3: Size Cleaning

In [4]:
def convert_size(size_str):
    if pd.isna(size_str) or 'Varies with device' in str(size_str):
        return np.nan
    size_str = str(size_str).upper().replace(',', '')
    if 'M' in size_str:
        return float(size_str.replace('M', ''))
    elif 'K' in size_str:
        return float(size_str.replace('K', '')) / 1024
    elif 'G' in size_str:
        return float(size_str.replace('G', '')) * 1024
    return np.nan

df['Size_MB'] = df['Size'].apply(convert_size)
size_median = df['Size_MB'].median()
df['Size_MB'] = df['Size_MB'].fillna(size_median)

print(f'Size median used: {size_median} MB')

Size median used: 10.0 MB


---
## Step 4: Installs Cleaning

In [5]:
installs_median = df['Minimum Installs'].median()

df['Installs_Numeric'] = (
    df['Installs'].str.replace('+', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .apply(pd.to_numeric, errors='coerce')
                  .fillna(installs_median)
)

print(f'Installs median used: {installs_median}')

Installs median used: 500.0


---
## Step 5: Currency Cleaning

In [6]:
currency_mode = df['Currency'].mode().iloc[0]
df['Currency'] = df['Currency'].fillna(currency_mode)

print(f'Currency mode used: {currency_mode}')

Currency mode used: USD


---
## Step 6: Final Check

In [7]:
if df.isnull().sum().sum() == 0:
    print('✅ ZERO NULLS — CLEAN DATASET')

print(f'Final shape: {df.shape}')

✅ ZERO NULLS — CLEAN DATASET
Final shape: (2312939, 26)
